In [ ]:
from pyspark import SparkContext

# =================================================================
# 1. Constantes 
# =================================================================

# Símbolos de Magnitudes y Estados
MAG_TEMP = 83
MAG_PREC = 89
FLAG_VALID = 'V'

# =================================================================
# 2. Extraccion de datos 
# =================================================================
sc = SparkContext.getOrCreate()
file_name = "calidad_aire_datos_meteo_mes.csv"
raw_rdd = sc.textFile(f"Sd_P1/{file_name}")

SEP = ';'

safe_raw_rdd = raw_rdd.filter(lambda line: len(line.split(SEP)) >= 9 and line.split(SEP)[3].isdigit())


parse_lambda = lambda line: (
    lambda parts: {
        'provincia': parts[0],
        'fecha': f"{parts[5]}-{parts[6].zfill(2)}-{parts[7].zfill(2)}",
        'muni': parts[1],
        'est': parts[2],
        'mag': int(parts[3]),
        
        'vals': [
            float(parts[8 + (i * 2)].replace(',', '.'))
            for i in range(24)
            if (8 + (i * 2) + 1) < len(parts) and 
               parts[8 + (i * 2) + 1] == FLAG_VALID
        ]
    }
)(line.split(SEP))

data_rdd = safe_raw_rdd.map(parse_lambda)

# =================================================================
# 3. Ejercicios 1 a 4
# =================================================================

# EJERCICIO 1
temp_validos_rdd = data_rdd.filter(lambda x: x['mag'] == MAG_TEMP and len(x['vals']) > 0)
total_validos = temp_validos_rdd.count()

# EJERCICIO 2
# Mappear por temperatura maxima por dias
final_daily_max = temp_validos_rdd.map(lambda x: (x['fecha'], max(x['vals']))) \
                                  .reduceByKey(lambda a, b: max(a, b))

# EJERCICIO 3
max_precip_por_dia = data_rdd.filter(lambda x: x['mag'] == MAG_PREC) \
                             .map(lambda x: (x['fecha'], (x['muni'], x['est'], sum(x['vals'])))) \
                             .reduceByKey(lambda x, y: x if x[2] >= y[2] else y)

# PARTE B: Máximo Absoluto del periodo
max_absoluto = max_precip_por_dia.max(key=lambda x: x[1][2])

# EJERCICIO 4
# Valor medio de las estaciones de municipios validos
get_daily_avg_lambda = lambda record: None if len(record['vals']) == 0 else (record['fecha'], sum(record['vals']) / len(record['vals']))

# 1. Se crean RDDs separados para cada estación específica
ref_station = temp_validos_rdd.filter(lambda x: int(x['muni']) == 6 and int(x['est']) == 4) \
                              .map(get_daily_avg_lambda).filter(lambda x: x is not None)

comp_station = temp_validos_rdd.filter(lambda x: int(x['muni']) == 5 and int(x['est']) == 2) \
                               .map(get_daily_avg_lambda).filter(lambda x: x is not None)
# 2. Unimos los datos por Fecha y calculamos sus porcentajes
percentage_rdd = ref_station.join(comp_station) \
                            .mapValues(lambda x: (x[1] / x[0]) * 100)

# =================================================================
# 4. Imprimir por pantalla
# =================================================================

combined_rdd = final_daily_max.fullOuterJoin(max_precip_por_dia).fullOuterJoin(percentage_rdd).sortByKey()
collected_data = combined_rdd.collect()

print(f"--- RESULTADOS ---")
print(f"[Ej 1] Total registros de temperatura válidos: {total_validos}")
print(f"[Ej 3B] Máx Absoluto Precipitación -> Fecha: {max_absoluto[0]} | Muni: {max_absoluto[1][0]} | Est: {max_absoluto[1][1]} | Total: {max_absoluto[1][2]} mm\n")

print(f"{'FECHA':<12} | {'T_MAX (Ej 2)':<15} | {'PRECIP MAX (Ej 3A) [Muni, Est, mm]':<36} | {'COMPARATIVA (Ej 4)':<18}")
print("-" * 92)

registro_variables = {}

for fecha, (datos_izq, pct) in collected_data:
    t_max = datos_izq[0] if datos_izq and datos_izq[0] is not None else "N/D"
    precip_data = datos_izq[1] if datos_izq and datos_izq[1] is not None else ("N/D", "N/D", "N/D")
    pct_str = f"{pct:.2f}%" if pct is not None else "N/D"
    
    registro_variables[fecha] = {
        'temp_max': t_max,
        'precip_max_muni': precip_data[0],
        'precip_max_est': precip_data[1],
        'precip_max_total': precip_data[2],
        'pct_comparativa': pct_str
    }
    
    print(f"{fecha:<12} | {str(t_max):<15} | {str(precip_data):<37}| {pct_str:<18}")


--- RESULTADOS ---
[Ej 1] Total registros de temperatura válidos: 224
[Ej 3B] Máx Absoluto Precipitación -> Fecha: 2026-02-05 | Muni: 115 | Est: 3 | Total: 30.8 mm

FECHA        | T_MAX (Ej 2)    | PRECIP MAX (Ej 3A) [Muni, Est, mm]   | COMPARATIVA (Ej 4)
--------------------------------------------------------------------------------------------
2026-02-01   | 11.7            | ('120', '1', 20.8)                   | 129.04%           
2026-02-02   | 12.2            | ('161', '1', 18.4)                   | 113.45%           
2026-02-03   | 9.8             | ('127', '4', 7.6)                    | 114.98%           
2026-02-04   | 11.3            | ('45', '2', 11.6)                    | 114.01%           
2026-02-05   | 15.5            | ('115', '3', 30.8)                   | 112.09%           
2026-02-06   | 10.3            | ('67', '1', 6.3)                     | 116.52%           
2026-02-07   | 9.0             | ('115', '3', 21.8)                   | 131.91%           
2026-02-08   |